In [ ]:
import sys, os

sys.path.append(os.path.abspath(".."))

from src.spanish_only import *

device = 'cpu'
source = 'validation'

Modelos probados:
- paraphrase-multilingual-mpnet-base-v2
- paraphrase-multilingual-MiniLM-L12-v2

He visto que algunas propuestas interesantes involucran finetunning contrastivo o combinación de resultados de varios modelos sin entrenamiento. Se me ocurre que igual puedo plantear una mezcla de ambos enfoques, haciendo el finetunning unicamente en español y esperando que esto mejore los resultados.


# Basic evaluation

In [ ]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
nickname = 'mpnet-base'

In [15]:
run_spanish_evaluation(model_name, nickname, device, source)

Loading Spanish data...
Encoding data: paraphrase-multilingual-mpnet-base-v2 on device: cpu


Batches: 100%|██████████| 146/146 [02:17<00:00,  1.06it/s]


Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed
Saving Spanish evaluation results...
Saved Spanish results to /home/david/Documents/Master/TFM/src/output/2025-11-19/005/results_spanish_monolingual.json
Updating Spanish ranking file...
Ranking español actualizado en /home/david/Documents/Master/TFM/src/output/ranking_spanish_validation.csv

Evaluación completada para paraphrase-multilingual-mpnet-base-v2
MAP español-español: 0.4167




# Training

## Preparacion de los datos

In [ ]:
import pandas as pd
import itertools
import re
import unicodedata

In [47]:
train_df = pd.read_csv('../data/training/spanish/taskA_training_es.tsv', sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

train_df[['jobtitle_1', 'jobtitle_2']]

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
20719,encargado de vestuarios,encargada de vestidores
20720,encargado de vestuarios/encargada de vestuarios,encargada de vestuarios
20721,acomodadora,acomodador/acomodadora
20722,acomodador,acomodador/acomodadora


In [48]:
# dividir las palabras con marcadores de genero, separados por /

expanded = []

for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']
    
    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]
    
    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                'jobtitle_1': a,
                'jobtitle_2': b
            })

df_expanded = pd.DataFrame(expanded)
df_expanded


,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
23868,encargado de vestuarios,encargada de vestidores
23869,encargado de vestuarios,encargada de vestuarios
23870,acomodadora,acomodador
23871,acomodador,acomodadora


In [49]:
# quedarnos solo con registros unicos, descartando los repetidos incluso cuando aparecen en la columna 1 y 2 intercambiados (ver ultimos tres registros del df_expanded)

df_norm = df_expanded.copy()

# Ordenar internamente cada pareja jobtitle_1 / jobtitle_2
df_norm[['jt1_sorted','jt2_sorted']] = (
    pd.DataFrame(
        df_norm.apply(lambda row: sorted([row['jobtitle_1'].strip(),
                                          row['jobtitle_2'].strip()]),
                      axis=1).to_list(),
        index=df_norm.index
    )
)

df_unique = df_norm.drop_duplicates(subset=['jt1_sorted','jt2_sorted'])
df_unique = df_unique[['jobtitle_1', 'jobtitle_2']].reset_index(drop=True)

df_unique

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
17882,encargada de vestidores,encargada de vestuarios
17883,encargado de vestuarios,encargada de vestuarios
17884,encargado de vestuarios,encargado de vestidores
17885,encargada de vestidores,encargado de vestuarios


## Entrenamiento del modelo

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# 3. Modelo y entrenamiento
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    evaluation_steps=500,
    output_path='./models/spanish-finetuned'
)

# 4. Evaluar con tu script existente
# python spanish_only.py --model ./models/spanish-finetuned --source validation